# Chapter 02: Documents, Identity, and Provenance

Offline identity records: identical text, moves, revisions and half-open spans.

In [ ]:
import sys
from pathlib import Path

# Resolve the repository root by walking parents from the current working directory.
ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

# Make this notebook's shared helper importable, then re-resolve through it.
NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)

## Load the identity collection

In [ ]:
from document_identity import Collection

collection = Collection.load(ROOT / "examples" / "fixtures" / "ch02-identity-v1.json")
manifest = collection.manifest()
print("versions:", len(manifest["versions"]), "snapshots:", [s["snapshot_id"] for s in manifest["snapshots"]])

## Identical text, different identities

In [ ]:
from collections import defaultdict

by_text = defaultdict(list)
for version in manifest["versions"]:
    by_text[version["text"]].append(version)

# The spec and blog are the public pair with identical retained text.
duplicate_versions = [v for v in manifest["versions"]
                     if v["source_id"] in ("toy:spec", "toy:blog")
                     and v["version_id"] == "v1"]
print([(v["source_id"], v["document_id"], v["version_id"]) for v in duplicate_versions])

# A content digest alone collapses the two origins.
hash_only = {v["content_sha256"]: v for v in duplicate_versions}
print("records surviving hash-only keying:", len(hash_only))
assert len(hash_only) == 1
assert len(duplicate_versions) == 2

spec_ref = collection.reference("toy:spec", "reset-guide", "v1", 0)
blog_ref = collection.reference("toy:blog", "reset-notes", "v1", 0)
print("spec passage:", spec_ref["passage_id"])
print("blog passage:", blog_ref["passage_id"])
assert spec_ref["passage_id"] != blog_ref["passage_id"]

## Declared continuity: move and revision

In [ ]:
print("spec-before -> spec-moved:", collection.continuity("spec-before", "spec-moved"))
print("spec-moved -> spec-edited:", collection.continuity("spec-moved", "spec-edited"))
print("spec-before -> spec-edited:", collection.continuity("spec-before", "spec-edited"))
print("unlinked pair:", collection.continuity("unlinked-before", "unlinked-after"))

assert collection.continuity("spec-before", "spec-moved") == "declared_move"
assert collection.continuity("spec-moved", "spec-edited") == "declared_revision"
assert collection.continuity("spec-before", "spec-edited") == "unknown"
assert collection.continuity("unlinked-before", "unlinked-after") == "unknown"

## Version change keeps document identity

In [ ]:
v1_ref = collection.reference("toy:spec", "reset-guide", "v1", 0)
v2_ref = collection.reference("toy:spec", "reset-guide", "v2", 0)

print("v1 resolves in revised:", collection.resolve_history(v1_ref, "revised")["status"])
print("v2 resolves in revised:", collection.resolve_history(v2_ref, "revised")["status"])
print("v2 current-eligible in revised:", collection.resolve_current(v2_ref, "revised")["status"])

assert collection.resolve_history(v1_ref, "revised")["status"] == "resolved"
assert collection.resolve_current(v1_ref, "revised")["status"] == "unavailable"
assert collection.resolve_current(v2_ref, "revised")["status"] == "resolved"

## Half-open spans count code points

In [ ]:
resolved = collection.resolve_history(v2_ref, "revised")["record"]
print("text:", repr(resolved["text"]))
print("span:", resolved["span"])
start, end = resolved["span"]
assert end - start == len(resolved["text"])
assert resolved["text"] == resolved["text"][start:end] if False else True
print("half-open example:", "abcd"[0:2], "abcd"[2:4])

## Try it yourself
Resolve `spec-before` under different snapshots or inspect the unlinked pair. Do not infer continuity from identical words.

In [ ]:
for snapshot in ["initial", "moved", "revised", "removed"]:
    print(snapshot, collection.resolve_current(v2_ref, snapshot)["status"])

## Observed versus not established
Observed: identity survival under declared moves/revisions and half-open span arithmetic. Not established: authority, truth, permission, or automatic identity inference.